# DuckDB

This notebook demonstrates how to use [DuckDB](https://duckdb.org/) to search public SRA metadata directly from the free [SRA metadata S3 bucket](https://registry.opendata.aws/ncbi-sra/).

**What is DuckDB?**

DuckDB is a small, fast database engine that runs right inside your notebook. You do not need to set up a database server, create an AWS account, or download large metadata files before asking questions.

Instead, DuckDB lets you query files where they already live.

DuckDB is useful for exploring SRA metadata before downloading sequence data. It helps you filter by dates, organisms, sequencing strategies, library attributes, sample details, and other run metadata. In this notebook, we will use it as a lightweight search tool for finding SRA accessions that match a specific research question.

This tool can be used on Windows, Mac or Linux. It works with Python, R, or the command line.

**Example:** For this demo, we will look up the accessions of `2025 whole-genome sequencing experiments` for the *Klebsiella* genus (it's a bacteria). The SRA metadata table can be filtered by date, by sequencing library attributes, and by sample/organism details.

---

## 1. Install DuckDB

Let's start by installing DuckDB into the Jupyter Environment. Outside of this exercise notebook, you will need to do this for your own machine using instruction present on the [DuckDB](https://duckdb.org/) website.

In [ ]:
# Install DuckDB into this Jupyter environment
%pip install duckdb

# For your own P
#pip install duckdb

**NOTE:** After installing DuckDB using this command, you will need to restart the Jupyter kernel using `Menu Bar > Kernel > Restart Kernel`. You may need to re-open this notebook in order to get it working.

## 2. Connect to DuckDB and Run a First Query

Now we will import DuckDB, create a local in-process connection, and use SQL to scan SRA metadata files directly from the public S3 bucket.

This example searches for 2025 whole-genome sequencing runs from the *Klebsiella* genus and returns a small set of matching accessions. The `LIMIT 100` keeps the demo output manageable while you are learning the query pattern.

In [ ]:
import duckdb

# Create an in-process
con = duckdb.connect()

# Run your query
result = con.sql("""
SELECT acc, experiment, sample_acc, sra_study, organism, center_name, mbytes
FROM read_parquet('s3://sra-pub-metadata-us-east-1/sra/metadata/*') -- parse parquet files stored on AWS ODP
WHERE organism ILIKE 'Klebsiella %' -- look for all organisms starting with Klebsiella
AND releasedate BETWEEN '2025-01-01' AND '2025-12-31'
AND librarysource='GENOMIC'
AND libraryselection='RANDOM'
AND assay_type = 'WGS'
LIMIT 100
""")
result

## 3. Refine the Results

The resulting python object can be filtered further with SQL by using it as a source table in an additional query.

The first query gives us a working set of matches. Because DuckDB returns a relation object, we can keep using SQL to filter those results further without rewriting the entire S3 query.

Here, we narrow the results to runs from one sequencing center.


In [ ]:
result = con.sql("SELECT * FROM result WHERE center_name='WELLCOME SANGER INSTITUTE'")
result

## 4. See What Metadata Fields Are Available

Before we search SRA metadata, it helps to know what fields are available.

The SRA metadata we are querying is stored in **Parquet files**. A Parquet file is a table-like data file, similar in spirit to a CSV file, but designed to work better with large datasets. It stores data efficiently and allows tools like DuckDB to read only the columns they need.

You do not need to open the Parquet file directly. DuckDB can inspect it for us.

A common first question is:

> “What columns can I search?”

For example, we may want to know whether the metadata includes fields like:

- organism name
- sequencing strategy
- assay type
- release date
- BioProject
- BioSample
- sequencing center

DuckDB’s `DESCRIBE` command lets us look at the structure of the metadata table before writing more specific queries.

In other words, this step helps us see the “menu” of available metadata fields.

In [ ]:
result = con.sql("DESCRIBE FROM read_parquet('s3://sra-pub-metadata-us-east-1/sra/metadata/*')").df()

result

## 5. Explore Values Before Filtering

Knowing the column names is only the first step.

Once we know that a field exists, we also need to know what kinds of values are stored in that field.

For example, a column might be called `organism`, but we may not know exactly how the organism names are written. Is it:

- `human`
- `Homo sapiens`
- `Homo sapiens sapiens`

The same issue can come up with fields like assay type, library strategy, sequencing platform, or sequencing center.

Before writing a filter, it is helpful to look at the unique values that actually appear in a column.

In SQL, we can use `SELECT DISTINCT` to do this.

`SELECT DISTINCT` means:

> “Show me each unique value in this column, but do not repeat duplicates.”

This helps us learn what values are available before we write a more specific query.

In [ ]:
result = con.execute("""
SELECT DISTINCT organism
FROM read_parquet('s3://sra-pub-metadata-us-east-1/sra/metadata/*')
WHERE releasedate>'2026-05-01'
AND organism ILIKE '%metagenome%' --look for any string containing the word metagenome
""").df()
result

## 6. Search Sample Attributes

So far, we have searched metadata fields that behave like regular table columns.

For example, a run may have top-level columns such as:

- `acc`
- `organism`
- `assay_type`
- `library_strategy`
- `releasedate`

But SRA records can also contain extra sample-specific details that do not always fit neatly into the same columns for every record.

These details are stored in the `attributes` column.

You can think of `attributes` as a collection of extra notes about the sample. Each attribute usually has a **name** and a **value**.

For example, one sample might include attributes like:

```text
collection_date: 2020
geo_loc_name: USA
isolation_source: soil
sequencing_method: 16S rRNA

In [ ]:
result = con.sql("""
-- first look for runs
WITH t AS (
  SELECT * FROM read_parquet('s3://sra-pub-metadata-us-east-1/sra/metadata/*')
  WHERE len(list_filter(attributes, x -> x.k = 'sequencing_method_sam' AND contains(x.v, '16S rRNA'))) > 0 -- check if list_filter is not empty
  LIMIT 10
  )
SELECT acc, organism, center_name, attr.k, attr.v
FROM t,UNNEST(attributes) AS t2(attr) -- cross join run metadata with sample metadata
""")
result

---

If you have any more specific questions, please visit the [DuckDB webpage](https://duckdb.org/) and the [DuckDB documentation repository](https://duckdb.org/docs/current/).

